<a href="https://colab.research.google.com/github/rahafabumwise/IEEE-AI-Modeling-Hackathon-2.0-Stage-1-Challenge/blob/main/%23%20Experiment%2001_Hidden%20operating%20regimes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final exact model — Kaggle score 0.20500

This notebook contains only the pipeline used by the successful submission:

- thermal-residual features
- missing-value indicators and median imputation
- stress engineered features
- CatBoost with 1840 iterations
- LightGBM with 1396 iterations
- 80% CatBoost + 20% LightGBM blend in log space

Run every cell from top to bottom..


In [1]:
!pip install -q catboost lightgbm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.0 MB/s eta 0:00:00


In [2]:
import os
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor


## 1. Load data

Change only `DATA_PATH` when needed.


In [3]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [4]:
DATA_PATH = "/content/drive/MyDrive/datasets/ieee-ai-modeling-hackathon2-stage-1-challenge"

train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
test = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

print("Train shape:", train.shape)
print("Test shape:", test.shape)

assert train.shape[0] == 24000
assert test.shape[0] == 16000
assert "edi" in train.columns
assert "id" in train.columns
assert "id" in test.columns


Train shape: (24000, 44)
Test shape: (16000, 43)


In [5]:
X_raw = train.drop(columns=["id", "edi"]).copy()
X_test_raw = test.drop(columns=["id"]).copy()

y_raw = train["edi"].copy()
y_log = np.log1p(y_raw)

print("X_raw:", X_raw.shape)
print("X_test_raw:", X_test_raw.shape)
print("y_log:", y_log.shape)


X_raw: (24000, 42)
X_test_raw: (16000, 42)
y_log: (24000,)


## 2. Exact preprocessing and feature engineering


In [6]:
MISSING_COLS = [
    "humidity",
    "sensor_17",
    "vibration_rms",
    "coolant_flow",
    "hours_since_service",
    "sensor_05"
]


def preprocess_fold(X_train_fold, X_valid_fold):
    X_train_fold = X_train_fold.copy()
    X_valid_fold = X_valid_fold.copy()

    for col in MISSING_COLS:
        X_train_fold[f"{col}_was_missing"] = (
            X_train_fold[col].isna().astype(int)
        )

        X_valid_fold[f"{col}_was_missing"] = (
            X_valid_fold[col].isna().astype(int)
        )

    X_train_fold["total_missing_count"] = (
        X_train_fold[MISSING_COLS].isna().sum(axis=1)
    )

    X_valid_fold["total_missing_count"] = (
        X_valid_fold[MISSING_COLS].isna().sum(axis=1)
    )

    fold_medians = {}

    for col in MISSING_COLS:
        median_value = X_train_fold[col].median()
        fold_medians[col] = median_value

        X_train_fold[col] = X_train_fold[col].fillna(median_value)
        X_valid_fold[col] = X_valid_fold[col].fillna(median_value)

    return X_train_fold, X_valid_fold, fold_medians


In [7]:
def add_engineered_features_stress(X):
    X = X.copy()

    X["load_duty_combo"] = (
        X["load_factor"] * X["duty_cycle"]
    )

    X["stress_index"] = (
        X["core_temp"]
        * X["load_factor"]
        * X["duty_cycle"]
    )

    X["thermal_excess"] = X["delta_ambient"]

    X["electrical_stress"] = (
        X["harmonic_thd"]
        * X["load_factor"]
    )

    X["mechanical_stress"] = (
        X["vibration_rms"]
        * X["load_factor"]
    )

    X["combined_operating_stress"] = (
        X["core_temp"]
        * X["load_factor"]
        * X["duty_cycle"]
        * (1 + X["harmonic_thd"])
    )

    return X


In [8]:
TEMP_PREDICTOR_FEATURES = [
    "asset_age",
    "load_factor",
    "duty_cycle",
    "coolant_flow",
    "humidity",
    "vibration_rms",
    "grid_freq",
    "line_voltage",
    "harmonic_thd",
    "phase_imbalance",
    "hours_since_service",
    "chamber_pressure"
]


def add_temperature_residual_features(
    X_train_raw,
    X_valid_raw
):
    X_train_raw = X_train_raw.copy()
    X_valid_raw = X_valid_raw.copy()

    temperature_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=10.0))
    ])

    temperature_model.fit(
        X_train_raw[TEMP_PREDICTOR_FEATURES],
        X_train_raw["core_temp"]
    )

    train_expected_temp = temperature_model.predict(
        X_train_raw[TEMP_PREDICTOR_FEATURES]
    )

    valid_expected_temp = temperature_model.predict(
        X_valid_raw[TEMP_PREDICTOR_FEATURES]
    )

    X_train_raw["expected_core_temp"] = train_expected_temp
    X_valid_raw["expected_core_temp"] = valid_expected_temp

    X_train_raw["core_temp_residual"] = (
        X_train_raw["core_temp"]
        - X_train_raw["expected_core_temp"]
    )

    X_valid_raw["core_temp_residual"] = (
        X_valid_raw["core_temp"]
        - X_valid_raw["expected_core_temp"]
    )

    X_train_raw["abs_core_temp_residual"] = (
        X_train_raw["core_temp_residual"].abs()
    )

    X_valid_raw["abs_core_temp_residual"] = (
        X_valid_raw["core_temp_residual"].abs()
    )

    return X_train_raw, X_valid_raw, temperature_model


In [32]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture


REGIME_FEATURES = [
    "load_factor",
    "core_temp",
    "vibration_rms",
    "grid_freq",
    "line_voltage",
    "chamber_pressure",
    "coolant_flow",
    "humidity",
    "duty_cycle",
    "harmonic_thd",
    "phase_imbalance",
    "delta_ambient"
]


def add_regime_probability_features(
    X_train_raw,
    X_valid_raw,
    n_regimes=3
):
    X_train_raw = X_train_raw.copy()
    X_valid_raw = X_valid_raw.copy()

    regime_imputer = SimpleImputer(
        strategy="median"
    )

    regime_scaler = StandardScaler()

    X_train_regime = regime_imputer.fit_transform(
        X_train_raw[REGIME_FEATURES]
    )

    X_valid_regime = regime_imputer.transform(
        X_valid_raw[REGIME_FEATURES]
    )

    X_train_regime = regime_scaler.fit_transform(
        X_train_regime
    )

    X_valid_regime = regime_scaler.transform(
        X_valid_regime
    )

    regime_model = GaussianMixture(
        n_components=n_regimes,
        covariance_type="full",
        random_state=42,
        n_init=5
    )

    regime_model.fit(
        X_train_regime
    )

    train_probabilities = regime_model.predict_proba(
        X_train_regime
    )

    valid_probabilities = regime_model.predict_proba(
        X_valid_regime
    )

    for regime_id in range(n_regimes):
        X_train_raw[
            f"regime_prob_{regime_id}"
        ] = train_probabilities[:, regime_id]

        X_valid_raw[
            f"regime_prob_{regime_id}"
        ] = valid_probabilities[:, regime_id]

    return (
        X_train_raw,
        X_valid_raw,
        regime_model,
        regime_imputer,
        regime_scaler
    )

In [46]:
def add_regime_interaction_features(X):
    X = X.copy()

    X["core_temp_regime_0"] = (
        X["core_temp"] * X["regime_prob_0"]
    )

    X["vibration_regime_0"] = (
        X["vibration_rms"] * X["regime_prob_0"]
    )

    X["load_regime_2"] = (
        X["load_factor"] * X["regime_prob_2"]
    )

    X["coolant_regime_2"] = (
        X["coolant_flow"] * X["regime_prob_2"]
    )

    return X

## 3. Build the exact final feature tables


In [49]:
X_full_train_raw = X_raw.copy()
X_full_test_raw = X_test_raw.copy()

# 1. Thermal features
(
    X_full_train_thermal,
    X_full_test_thermal,
    final_temperature_model
) = add_temperature_residual_features(
    X_full_train_raw,
    X_full_test_raw
)

# 2. Regime probabilities
(
    X_full_train_regime,
    X_full_test_regime,
    final_regime_model,
    final_regime_imputer,
    final_regime_scaler
) = add_regime_probability_features(
    X_full_train_thermal,
    X_full_test_thermal,
    n_regimes=3
)

# 3. Missing-value preprocessing
(
    X_full_train,
    X_full_test,
    full_medians
) = preprocess_fold(
    X_full_train_regime,
    X_full_test_regime
)

# 4. Existing stress features
X_full_train = add_engineered_features_stress(
    X_full_train
)

X_full_test = add_engineered_features_stress(
    X_full_test
)

# 5. NEW regime interaction features
X_full_train = add_regime_interaction_features(
    X_full_train
)

X_full_test = add_regime_interaction_features(
    X_full_test
)

# Checks
assert list(X_full_train.columns) == list(
    X_full_test.columns
)

assert X_full_train.isna().sum().sum() == 0
assert X_full_test.isna().sum().sum() == 0

print("Final training shape:", X_full_train.shape)
print("Final test shape:", X_full_test.shape)

Final training shape: (24000, 65)
Final test shape: (16000, 65)


In [50]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_log_error

def calculate_rmsle(actual, prediction):
    prediction = np.clip(
        np.asarray(prediction),
        0,
        None
    )

    return np.sqrt(
        mean_squared_log_error(
            np.asarray(actual),
            prediction
        )
    )

In [51]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fixed_folds = list(cv.split(X_raw))

In [52]:
regime_cat_oof_log = np.zeros(len(X_raw))
regime_cat_fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(
    fixed_folds,
    start=1
):
    print(f"\n===== Regime CatBoost Fold {fold} =====")

    X_train_fold_raw = X_raw.iloc[train_idx].copy()
    X_valid_fold_raw = X_raw.iloc[valid_idx].copy()

    y_train_fold = y_log.iloc[train_idx]
    y_valid_fold = y_log.iloc[valid_idx]

    # 1. Thermal features
    (
        X_train_fold_raw,
        X_valid_fold_raw,
        fold_temperature_model
    ) = add_temperature_residual_features(
        X_train_fold_raw,
        X_valid_fold_raw
    )

    # 2. Regime probabilities
    (
        X_train_fold_raw,
        X_valid_fold_raw,
        fold_regime_model,
        fold_regime_imputer,
        fold_regime_scaler
    ) = add_regime_probability_features(
        X_train_fold_raw,
        X_valid_fold_raw,
        n_regimes=3
    )

    # 3. Missing-value preprocessing
    (
        X_train_fold,
        X_valid_fold,
        fold_medians
    ) = preprocess_fold(
        X_train_fold_raw,
        X_valid_fold_raw
    )

    # 4. Stress features
    X_train_fold = add_engineered_features_stress(
        X_train_fold
    )

    X_valid_fold = add_engineered_features_stress(
        X_valid_fold
    )

    # 5. Regime interaction features
    X_train_fold = add_regime_interaction_features(
        X_train_fold
    )

    X_valid_fold = add_regime_interaction_features(
        X_valid_fold
    )

    # Safety checks
    assert list(X_train_fold.columns) == list(
        X_valid_fold.columns
    )

    assert X_train_fold.isna().sum().sum() == 0
    assert X_valid_fold.isna().sum().sum() == 0

    print(
        "Fold feature count:",
        X_train_fold.shape[1]
    )

    # 6. Train CatBoost
    model = CatBoostRegressor(
        iterations=2000,
        depth=6,
        learning_rate=0.03,
        l2_leaf_reg=3.0,
        loss_function="RMSE",
        random_seed=42,
        verbose=0,
        early_stopping_rounds=100
    )

    model.fit(
        X_train_fold,
        y_train_fold,
        eval_set=(X_valid_fold, y_valid_fold),
        use_best_model=True,
        verbose=False
    )

    # 7. Predict validation fold
    fold_pred_log = model.predict(
        X_valid_fold
    )

    regime_cat_oof_log[valid_idx] = fold_pred_log

    fold_pred = np.clip(
        np.expm1(fold_pred_log),
        0,
        None
    )

    fold_actual = np.expm1(
        y_valid_fold
    )

    fold_score = calculate_rmsle(
        fold_actual,
        fold_pred
    )

    regime_cat_fold_scores.append(
        fold_score
    )

    print("Fold RMSLE:", fold_score)


===== Regime CatBoost Fold 1 =====
Fold feature count: 65
Fold RMSLE: 0.19401066417494076

===== Regime CatBoost Fold 2 =====
Fold feature count: 65
Fold RMSLE: 0.18833067365734588

===== Regime CatBoost Fold 3 =====
Fold feature count: 65
Fold RMSLE: 0.18946377232736228

===== Regime CatBoost Fold 4 =====
Fold feature count: 65
Fold RMSLE: 0.18824114575481304

===== Regime CatBoost Fold 5 =====
Fold feature count: 65
Fold RMSLE: 0.1917350796823901


In [53]:
regime_cat_oof_pred = np.clip(
    np.expm1(regime_cat_oof_log),
    0,
    None
)

regime_cat_global = calculate_rmsle(
    train["edi"].values,
    regime_cat_oof_pred
)

print("Regime CatBoost OOF:", regime_cat_global)
print("Fold scores:", regime_cat_fold_scores)

Regime CatBoost OOF: 0.19036920508015112
Fold scores: [np.float64(0.19401066417494076), np.float64(0.18833067365734588), np.float64(0.18946377232736228), np.float64(0.18824114575481304), np.float64(0.1917350796823901)]


In [54]:
import lightgbm as lgb
from lightgbm import LGBMRegressor

regime_lgb_oof_log = np.zeros(len(X_raw))
regime_lgb_fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(
    fixed_folds,
    start=1
):
    print(f"\n===== Regime LightGBM Fold {fold} =====")

    # Raw fold data
    X_train_fold_raw = X_raw.iloc[train_idx].copy()
    X_valid_fold_raw = X_raw.iloc[valid_idx].copy()

    y_train_fold = y_log.iloc[train_idx]
    y_valid_fold = y_log.iloc[valid_idx]

    # 1. Thermal residual features
    (
        X_train_fold_raw,
        X_valid_fold_raw,
        fold_temperature_model
    ) = add_temperature_residual_features(
        X_train_fold_raw,
        X_valid_fold_raw
    )

    # 2. Regime probability features
    (
        X_train_fold_raw,
        X_valid_fold_raw,
        fold_regime_model,
        fold_regime_imputer,
        fold_regime_scaler
    ) = add_regime_probability_features(
        X_train_fold_raw,
        X_valid_fold_raw,
        n_regimes=3
    )

    # 3. Missing-value preprocessing
    (
        X_train_fold,
        X_valid_fold,
        fold_medians
    ) = preprocess_fold(
        X_train_fold_raw,
        X_valid_fold_raw
    )

    # 4. Existing stress features
    X_train_fold = add_engineered_features_stress(
        X_train_fold
    )

    X_valid_fold = add_engineered_features_stress(
        X_valid_fold
    )

    # 5. New regime interaction features
    X_train_fold = add_regime_interaction_features(
        X_train_fold
    )

    X_valid_fold = add_regime_interaction_features(
        X_valid_fold
    )

    # Safety checks
    assert list(X_train_fold.columns) == list(
        X_valid_fold.columns
    )

    assert X_train_fold.isna().sum().sum() == 0
    assert X_valid_fold.isna().sum().sum() == 0

    print(
        "LightGBM fold feature count:",
        X_train_fold.shape[1]
    )

    # 6. Train LightGBM
    model = LGBMRegressor(
        objective="regression",
        n_estimators=4000,
        learning_rate=0.02,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=30,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    model.fit(
        X_train_fold,
        y_train_fold,
        eval_set=[
            (X_valid_fold, y_valid_fold)
        ],
        eval_metric="rmse",
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=100,
                verbose=False
            ),
            lgb.log_evaluation(
                period=0
            )
        ]
    )

    # 7. Predict validation fold
    fold_pred_log = model.predict(
        X_valid_fold,
        num_iteration=model.best_iteration_
    )

    regime_lgb_oof_log[valid_idx] = fold_pred_log

    fold_pred = np.clip(
        np.expm1(fold_pred_log),
        0,
        None
    )

    fold_actual = np.expm1(
        y_valid_fold
    )

    fold_score = calculate_rmsle(
        fold_actual,
        fold_pred
    )

    regime_lgb_fold_scores.append(
        fold_score
    )

    print("Best iteration:", model.best_iteration_)
    print("Fold RMSLE:", fold_score)


# Overall LightGBM OOF result
regime_lgb_oof_pred = np.clip(
    np.expm1(regime_lgb_oof_log),
    0,
    None
)

regime_lgb_global = calculate_rmsle(
    train["edi"].values,
    regime_lgb_oof_pred
)

print("\nRegime interaction LightGBM OOF:", regime_lgb_global)
print("Fold scores:", regime_lgb_fold_scores)


===== Regime LightGBM Fold 1 =====
LightGBM fold feature count: 65
Best iteration: 1368
Fold RMSLE: 0.20239782543847712

===== Regime LightGBM Fold 2 =====
LightGBM fold feature count: 65
Best iteration: 1537
Fold RMSLE: 0.19500558708677884

===== Regime LightGBM Fold 3 =====
LightGBM fold feature count: 65
Best iteration: 1145
Fold RMSLE: 0.19364900680548183

===== Regime LightGBM Fold 4 =====
LightGBM fold feature count: 65
Best iteration: 1501
Fold RMSLE: 0.19580919235520958

===== Regime LightGBM Fold 5 =====
LightGBM fold feature count: 65
Best iteration: 1236
Fold RMSLE: 0.19860222033187308

Regime interaction LightGBM OOF: 0.19711726373181138
Fold scores: [np.float64(0.20239782543847712), np.float64(0.19500558708677884), np.float64(0.19364900680548183), np.float64(0.19580919235520958), np.float64(0.19860222033187308)]


In [55]:
regime_lgb_oof_pred = np.clip(
    np.expm1(regime_lgb_oof_log),
    0,
    None
)

regime_lgb_global = calculate_rmsle(
    train["edi"].values,
    regime_lgb_oof_pred
)

print("Regime LightGBM OOF:", regime_lgb_global)
print("Fold scores:", regime_lgb_fold_scores)

Regime LightGBM OOF: 0.19711726373181138
Fold scores: [np.float64(0.20239782543847712), np.float64(0.19500558708677884), np.float64(0.19364900680548183), np.float64(0.19580919235520958), np.float64(0.19860222033187308)]


In [56]:
regime_blend_oof_log = (
    0.80 * regime_cat_oof_log
    + 0.20 * regime_lgb_oof_log
)

regime_blend_oof_pred = np.clip(
    np.expm1(regime_blend_oof_log),
    0,
    None
)

regime_blend_global = calculate_rmsle(
    train["edi"].values,
    regime_blend_oof_pred
)

print("Baseline blend OOF: 0.189636")
print("Regime blend OOF:", regime_blend_global)
print(
    "Difference:",
    regime_blend_global - 0.189636
)

Baseline blend OOF: 0.189636
Regime blend OOF: 0.18984287704428882
Difference: 0.00020687704428881681


## 4. Train the exact final CatBoost model

The original successful notebook used **1840 iterations**.


In [10]:
final_thermal_model = CatBoostRegressor(
    iterations=1840,
    depth=6,
    learning_rate=0.03,
    l2_leaf_reg=3.0,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

final_thermal_model.fit(
    X_full_train,
    y_log
)

thermal_test_log = final_thermal_model.predict(
    X_full_test
)

print("CatBoost predictions:", len(thermal_test_log))
print("CatBoost missing:", np.isnan(thermal_test_log).sum())
print("CatBoost infinite:", np.isinf(thermal_test_log).sum())


0:	learn: 0.9283741	total: 73.3ms	remaining: 2m 14s
100:	learn: 0.2744276	total: 1.99s	remaining: 34.2s
200:	learn: 0.2143198	total: 3.92s	remaining: 31.9s
300:	learn: 0.1983686	total: 5.75s	remaining: 29.4s
400:	learn: 0.1901752	total: 7.57s	remaining: 27.2s
500:	learn: 0.1843760	total: 10.6s	remaining: 28.4s
600:	learn: 0.1801740	total: 14s	remaining: 28.9s
700:	learn: 0.1767324	total: 16s	remaining: 26s
800:	learn: 0.1737308	total: 17.8s	remaining: 23.1s
900:	learn: 0.1710598	total: 19.6s	remaining: 20.5s
1000:	learn: 0.1685288	total: 21.5s	remaining: 18s
1100:	learn: 0.1661713	total: 23.3s	remaining: 15.6s
1200:	learn: 0.1638472	total: 25.7s	remaining: 13.7s
1300:	learn: 0.1616742	total: 29s	remaining: 12s
1400:	learn: 0.1595701	total: 31s	remaining: 9.7s
1500:	learn: 0.1575931	total: 32.8s	remaining: 7.4s
1600:	learn: 0.1555828	total: 34.6s	remaining: 5.17s
1700:	learn: 0.1537211	total: 36.4s	remaining: 2.98s
1800:	learn: 0.1518733	total: 38.3s	remaining: 829ms
1839:	learn: 0.1511

## 5. Train the exact final LightGBM model

The original successful notebook used **1396 iterations**.


In [34]:
thermal_lgb_final_model = LGBMRegressor(
    objective="regression",
    n_estimators=1396,
    learning_rate=0.02,

    num_leaves=31,
    max_depth=-1,
    min_child_samples=30,

    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.85,

    reg_alpha=0.1,
    reg_lambda=1.0,

    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

thermal_lgb_final_model.fit(
    X_full_train,
    y_log
)

thermal_lgb_test_log = thermal_lgb_final_model.predict(
    X_full_test
)

print("LightGBM predictions:", len(thermal_lgb_test_log))
print("LightGBM missing:", np.isnan(thermal_lgb_test_log).sum())
print("LightGBM infinite:", np.isinf(thermal_lgb_test_log).sum())


LightGBM predictions: 16000
LightGBM missing: 0
LightGBM infinite: 0


## 6. Exact 80/20 log-space blend


In [35]:
thermal_blend_test_log = (
    0.80 * thermal_test_log
    + 0.20 * thermal_lgb_test_log
)

thermal_blend_test_pred = np.clip(
    np.expm1(thermal_blend_test_log),
    0,
    None
)


## 7. Create the submission


In [36]:
submission_thermal_blend = pd.DataFrame({
    "id": test["id"].values,
    "edi": thermal_blend_test_pred
})

submission_thermal_blend.to_csv(
    "submission_thermal_cat80_lgb20.csv",
    index=False
)

print("Submission shape:", submission_thermal_blend.shape)
print(
    "Missing predictions:",
    submission_thermal_blend["edi"].isna().sum()
)
print(
    "Infinite predictions:",
    np.isinf(submission_thermal_blend["edi"]).sum()
)
print(
    "Duplicate IDs:",
    submission_thermal_blend["id"].duplicated().sum()
)
print(
    "Minimum prediction:",
    submission_thermal_blend["edi"].min()
)
print(
    "Maximum prediction:",
    submission_thermal_blend["edi"].max()
)

display(submission_thermal_blend.head())

assert submission_thermal_blend.shape == (16000, 2)
assert submission_thermal_blend["edi"].isna().sum() == 0
assert np.isfinite(submission_thermal_blend["edi"]).all()
assert submission_thermal_blend["id"].equals(test["id"])


Submission shape: (16000, 2)
Missing predictions: 0
Infinite predictions: 0
Duplicate IDs: 0
Minimum prediction: 3.9497011483660285
Maximum prediction: 734.404715092106


,id,edi
0,24000,46.211731
1,24001,38.583051
2,24002,51.464928
3,24003,77.873571
4,24004,31.500393


The original successful run produced approximately:

- minimum prediction: `3.9652839915`
- maximum prediction: `737.3295358423`
- first prediction: `46.171586`

Small floating-point differences can occur across library versions.


In [14]:
from google.colab import files
files.download("submission_thermal_cat80_lgb20.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Experiment 01 — Hidden operating regimes

In [15]:
REGIME_FEATURES = [
    "load_factor",
    "core_temp",
    "vibration_rms",
    "grid_freq",
    "line_voltage",
    "chamber_pressure",
    "coolant_flow",
    "humidity",
    "duty_cycle",
    "harmonic_thd",
    "phase_imbalance",
    "delta_ambient"
]

In [16]:
X_regime_train = train[REGIME_FEATURES].copy()
X_regime_test = test[REGIME_FEATURES].copy()

print("Train regime shape:", X_regime_train.shape)
print("Test regime shape:", X_regime_test.shape)

display(X_regime_train.head())

Train regime shape: (24000, 12)
Test regime shape: (16000, 12)


,load_factor,core_temp,vibration_rms,grid_freq,line_voltage,chamber_pressure,coolant_flow,humidity,duty_cycle,harmonic_thd,phase_imbalance,delta_ambient
0,12.55941,66.28937,0.59021,49.70432,398.45960,2.18900,8.87366,NaN,0.77936,4.74025,1.94691,41.32948
1,4.38646,50.11278,1.82394,49.83773,399.37172,1.94912,8.30789,39.74295,0.93859,3.86965,0.65556,20.89554
2,7.68447,59.73105,0.83965,49.87547,416.54405,4.17573,6.02931,81.95129,0.91610,4.19463,0.45950,47.49396
3,6.05584,61.48525,0.14636,49.89966,406.26383,5.98682,7.39218,45.97983,0.56353,3.37983,2.10357,46.06761
4,10.45765,60.96648,2.12707,49.64769,413.64154,5.62247,7.74634,57.17967,0.71667,0.35512,0.66403,43.21165


In [17]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

regime_imputer = SimpleImputer(
    strategy="median"
)

regime_scaler = StandardScaler()

X_regime_train_imputed = regime_imputer.fit_transform(
    X_regime_train
)

X_regime_test_imputed = regime_imputer.transform(
    X_regime_test
)

X_regime_train_scaled = regime_scaler.fit_transform(
    X_regime_train_imputed
)

X_regime_test_scaled = regime_scaler.transform(
    X_regime_test_imputed
)

print(
    "Missing after preprocessing:",
    np.isnan(X_regime_train_scaled).sum()
)

print(
    "Test missing after preprocessing:",
    np.isnan(X_regime_test_scaled).sum()
)

Missing after preprocessing: 0
Test missing after preprocessing: 0


In [18]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

In [19]:
gmm_results = []

for n_regimes in range(2, 7):

    gmm = GaussianMixture(
        n_components=n_regimes,
        covariance_type="full",
        random_state=42,
        n_init=5
    )

    train_regime_id = gmm.fit_predict(
        X_regime_train_scaled
    )

    bic = gmm.bic(
        X_regime_train_scaled
    )

    aic = gmm.aic(
        X_regime_train_scaled
    )

    silhouette = silhouette_score(
        X_regime_train_scaled,
        train_regime_id
    )

    cluster_sizes = np.bincount(
        train_regime_id
    )

    gmm_results.append({
        "n_regimes": n_regimes,
        "bic": bic,
        "aic": aic,
        "silhouette": silhouette,
        "smallest_cluster": cluster_sizes.min(),
        "largest_cluster": cluster_sizes.max()
    })

gmm_results = pd.DataFrame(
    gmm_results
)

display(
    gmm_results.sort_values("bic")
)

,n_regimes,bic,aic,silhouette,smallest_cluster,largest_cluster
4,6,755260.845164,750854.079199,0.027017,2525,5502
3,5,755781.321347,752110.364011,0.031191,3202,6223
2,4,756640.038717,753704.890011,0.039674,3647,8594
1,3,758902.805080,756703.465003,0.064782,5935,11765
0,2,762077.745190,760614.213741,0.083410,9196,14804


In [20]:
from sklearn.cluster import KMeans

kmeans_results = []

for n_regimes in range(2, 7):

    kmeans = KMeans(
        n_clusters=n_regimes,
        random_state=42,
        n_init=20
    )

    train_regime_id = kmeans.fit_predict(
        X_regime_train_scaled
    )

    silhouette = silhouette_score(
        X_regime_train_scaled,
        train_regime_id
    )

    cluster_sizes = np.bincount(
        train_regime_id
    )

    kmeans_results.append({
        "n_regimes": n_regimes,
        "silhouette": silhouette,
        "smallest_cluster": cluster_sizes.min(),
        "largest_cluster": cluster_sizes.max()
    })

kmeans_results = pd.DataFrame(
    kmeans_results
)

display(
    kmeans_results.sort_values(
        "silhouette",
        ascending=False
    )
)

,n_regimes,silhouette,smallest_cluster,largest_cluster
0,2,0.111929,11106,12894
1,3,0.097682,5216,10330
4,6,0.071495,2501,5408
2,4,0.069277,4412,7012
3,5,0.069223,3332,5951


In [21]:
candidate_regimes = 3

gmm_candidate = GaussianMixture(
    n_components=candidate_regimes,
    covariance_type="full",
    random_state=42,
    n_init=5
)

train_regime_id = gmm_candidate.fit_predict(
    X_regime_train_scaled
)

test_regime_id = gmm_candidate.predict(
    X_regime_test_scaled
)

train_regime_probabilities = (
    gmm_candidate.predict_proba(
        X_regime_train_scaled
    )
)

test_regime_probabilities = (
    gmm_candidate.predict_proba(
        X_regime_test_scaled
    )
)

In [22]:
regime_analysis_train = train.copy()
regime_analysis_test = test.copy()

regime_analysis_train["regime_id"] = (
    train_regime_id
)

regime_analysis_test["regime_id"] = (
    test_regime_id
)

In [23]:
train_regime_counts = (
    regime_analysis_train["regime_id"]
    .value_counts()
    .sort_index()
)

test_regime_counts = (
    regime_analysis_test["regime_id"]
    .value_counts()
    .sort_index()
)

regime_size_comparison = pd.DataFrame({
    "train_count": train_regime_counts,
    "test_count": test_regime_counts
})

regime_size_comparison["train_percent"] = (
    regime_size_comparison["train_count"]
    / len(train)
    * 100
)

regime_size_comparison["test_percent"] = (
    regime_size_comparison["test_count"]
    / len(test)
    * 100
)

regime_size_comparison["percentage_shift"] = (
    regime_size_comparison["test_percent"]
    - regime_size_comparison["train_percent"]
)

display(regime_size_comparison)

,train_count,test_count,train_percent,test_percent,percentage_shift
regime_id,,,,,
0,6300,4595,26.250000,28.71875,2.468750
1,11765,7050,49.020833,44.06250,-4.958333
2,5935,4355,24.729167,27.21875,2.489583


In [24]:
regime_feature_summary = (
    regime_analysis_train
    .groupby("regime_id")[
        REGIME_FEATURES
    ]
    .mean()
    .round(3)
)

display(regime_feature_summary)

,load_factor,core_temp,vibration_rms,grid_freq,line_voltage,chamber_pressure,coolant_flow,humidity,duty_cycle,harmonic_thd,phase_imbalance,delta_ambient
regime_id,,,,,,,,,,,,
0,5.972,61.099,1.592,49.901,406.777,4.781,7.653,47.827,0.622,3.480,1.536,40.120
1,5.327,58.422,0.606,49.901,408.327,3.375,7.231,47.783,0.610,3.057,1.005,37.408
2,9.653,58.846,1.046,49.898,408.151,3.552,12.192,47.535,0.620,3.206,0.920,37.855


In [25]:
regime_feature_medians = (
    regime_analysis_train
    .groupby("regime_id")[
        REGIME_FEATURES
    ]
    .median()
    .round(3)
)

display(regime_feature_medians)

,load_factor,core_temp,vibration_rms,grid_freq,line_voltage,chamber_pressure,coolant_flow,humidity,duty_cycle,harmonic_thd,phase_imbalance,delta_ambient
regime_id,,,,,,,,,,,,
0,5.643,61.191,1.589,49.901,406.748,4.628,7.508,47.856,0.638,3.472,1.351,39.979
1,5.120,58.271,0.559,49.902,408.392,3.298,7.166,47.861,0.626,3.030,0.893,37.320
2,9.748,58.783,1.012,49.897,408.175,3.406,12.103,47.939,0.636,3.158,0.783,37.851


In [26]:
regime_target_summary = (
    regime_analysis_train
    .groupby("regime_id")["edi"]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max"
    ])
)

display(regime_target_summary)

,count,mean,median,std,min,max
regime_id,,,,,,
0,6300,95.165698,62.2832,95.062340,4.0868,796.3415
1,11765,64.851153,40.8868,67.723007,3.3621,729.6684
2,5935,91.700246,62.0634,88.974118,4.2813,673.6406


In [27]:
regime_analysis_train["log_edi"] = np.log1p(
    regime_analysis_train["edi"]
)

regime_log_target_summary = (
    regime_analysis_train
    .groupby("regime_id")["log_edi"]
    .agg([
        "mean",
        "median",
        "std"
    ])
)

display(regime_log_target_summary)

,mean,median,std
regime_id,,,
0,4.141187,4.147620,0.940854
1,3.762020,3.734971,0.927964
2,4.121899,4.144141,0.924986


In [28]:
relationship_features = [
    "asset_age",
    "core_temp",
    "load_factor",
    "vibration_rms",
    "coolant_flow",
    "duty_cycle",
    "harmonic_thd",
    "hours_since_service"
]

In [29]:
regime_correlations = []

for regime_id in sorted(
    regime_analysis_train["regime_id"].unique()
):

    subset = regime_analysis_train[
        regime_analysis_train["regime_id"]
        == regime_id
    ]

    for feature in relationship_features:

        correlation = subset[
            [feature, "log_edi"]
        ].corr().iloc[0, 1]

        regime_correlations.append({
            "regime_id": regime_id,
            "feature": feature,
            "correlation_with_log_edi": correlation
        })

regime_correlations = pd.DataFrame(
    regime_correlations
)

display(
    regime_correlations.pivot(
        index="feature",
        columns="regime_id",
        values="correlation_with_log_edi"
    ).round(3)
)

regime_id,0,1,2
feature,,,
asset_age,0.543,0.632,0.572
coolant_flow,0.121,0.038,-0.049
core_temp,0.552,0.410,0.436
duty_cycle,0.205,0.169,0.233
harmonic_thd,0.321,0.230,0.237
hours_since_service,0.186,0.271,0.226
load_factor,0.248,0.168,0.282
vibration_rms,0.244,0.125,0.129


In [30]:
max_regime_probability = (
    train_regime_probabilities.max(axis=1)
)

confidence_summary = pd.Series(
    max_regime_probability
).describe(
    percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]
)

display(confidence_summary)

print(
    "Rows with confidence above 0.80:",
    (max_regime_probability >= 0.80).mean()
)

print(
    "Rows with confidence below 0.50:",
    (max_regime_probability < 0.50).mean()
)

,0
count,24000.000000
mean,0.803167
std,0.164725
min,0.337954
10%,0.542840
25%,0.678969
50%,0.851525
75%,0.943339
90%,0.987706
max,1.000000


Rows with confidence above 0.80: 0.5881666666666666
Rows with confidence below 0.50: 0.049916666666666665


In [31]:
missing_by_regime = pd.DataFrame({
    "regime_id": train_regime_id
})

for col in MISSING_COLS:
    missing_by_regime[f"{col}_missing"] = (
        train[col].isna().astype(int)
    )

missing_by_regime_summary = (
    missing_by_regime
    .groupby("regime_id")
    .mean()
    .mul(100)
    .round(2)
)

display(missing_by_regime_summary)

,humidity_missing,sensor_17_missing,vibration_rms_missing,coolant_flow_missing,hours_since_service_missing,sensor_05_missing
regime_id,,,,,,
0,7.62,7.95,2.67,6.33,3.73,3.73
1,8.03,7.66,6.10,6.42,3.79,3.87
2,7.90,8.61,4.48,4.01,3.17,3.71
